# 03 — Build the feature table

**What this notebook is for.** Turn clean Parquet into the table the models train on:
one row per (flood station, 15-minute anchor), carrying features from the past and labels
from the future.

**The single rule.** A feature at time *t* may only use data recorded at or before *t*.
Break it and you get a wonderful validation score and a model that fails on its first
real day. Every rolling window in `src/bkkflood/features.py` is trailing; every lag is
backwards; every label window starts at *t+1*.

**Two optional inputs** are joined if they exist and skipped silently if not:

* `forecast_rain.parquet` — rain that has not fallen yet (notebook 03b). Measured to be
  about three times more predictive of a 6-hour flood than past rain.
* `station_spatial.parquet` — terrain from the DEM (notebook 03c). Tested weak so far, for
  a reason worth knowing: 31 m SRTM cannot see a 40 cm dip in a road.

**Runtime:** roughly 10-20 minutes per year, mostly spent on the water dataset.

In [1]:
import sys, pathlib
# Make the shared library importable no matter where Jupyter was started from.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "config" / "config.yaml").is_file())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

from bkkflood import CFG, PATHS
print("project root:", ROOT)
print("config version:", CFG["project"]["version"])
from bkkflood import features
from bkkflood.features import build_year, feature_columns, label_columns
from bkkflood.labels import describe_labels

project root: /Users/pritimmondal/Projects/bkk-flood-forecast
config version: 2.0.0


## 1. Build one year first and inspect it

Never launch a seven-year run before looking at one year's output. Most mistakes are
obvious in the first table and invisible in a progress bar.

In [2]:
sample = build_year(2023)
print(f"{len(sample):,} rows x {sample.shape[1]} columns")
sample.head()

[2023] loading flood depths ...


[2023] building rain / water / flow aggregates ...


[2023] 3,370,379 rows, y_ge15_1h positives: 823
3,370,379 rows x 70 columns


,station_code,site_timestamp,y_maxdepth_1h,y_valid_1h,y_ge5_1h,y_ge15_1h,y_ge30_1h,y_maxdepth_3h,y_valid_3h,y_ge5_3h,y_ge15_3h,y_ge30_3h,y_maxdepth_6h,y_valid_6h,y_ge5_6h,y_ge15_6h,y_ge30_6h,fl_depth_now,fl_depth_lag1h,fl_depth_lag3h,fl_rise_15min,fl_rise_1h,fl_max3h,fl_max24h,fl_mean1h,fl_std3h,fl_hrs_since_5cm,fl_hrs_since_15cm,fl_missing_share3h,prefix,...,rain_antecedent_ratio,water_rise1h_mean,water_rise1h_max,water_rise3h_mean,water_rising_share,water_offline_share,flow_mean,flow_velocity_mean,flow_negative_share,flow_offline_share,rain_fcst_1h,rain_fcst_3h,rain_fcst_6h,elev_m,slope_deg,depression_m,elev_rank_local,dist_water_km,cal_hour_sin,cal_hour_cos,cal_doy_sin,cal_doy_cos,cal_monsoon,tide_m2_sin,tide_m2_cos,tide_spring_neap,rain_x_recent_flood,is_onset_ge5,is_onset_ge15,is_onset_ge30
0,FL.BBN.01,2023-01-01 00:00:00,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,72.0,168.0,0.0,BBN,...,0.0,NaN,NaN,NaN,0.000000,0.056604,1.557037,0.047778,0.0,0.068966,0.0,0.0,0.0,4.0,2.936,11.0,0.316,0.0,0.000000,1.000000,0.017202,0.999852,0,0.321101,0.947045,0.161987,0.0,1,1,1
1,FL.BBN.01,2023-01-01 00:15:00,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,72.0,168.0,0.0,BBN,...,0.0,NaN,NaN,NaN,0.000000,0.056604,1.692222,0.049630,0.0,0.068966,0.0,0.0,0.0,4.0,2.936,11.0,0.316,0.0,0.065403,0.997859,0.017202,0.999852,0,0.437988,0.898981,0.159800,0.0,1,1,1
2,FL.BBN.01,2023-01-01 00:30:00,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,72.0,168.0,0.0,BBN,...,0.0,NaN,NaN,NaN,0.000000,0.056604,1.953704,0.060000,0.0,0.068966,0.0,0.0,0.0,4.0,2.936,11.0,0.316,0.0,0.130526,0.991445,0.017202,0.999852,0,0.547878,0.836558,0.157612,0.0,1,1,1
3,FL.BBN.01,2023-01-01 00:45:00,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,NaN,NaN,0.0,NaN,0.0,0.0,0.0,0.0,72.0,168.0,0.0,BBN,...,0.0,NaN,NaN,NaN,0.000000,0.056604,2.316296,0.134444,0.0,0.068966,0.0,0.0,0.0,4.0,2.936,11.0,0.316,0.0,0.195090,0.980785,0.017202,0.999852,0,0.649018,0.760773,0.155423,0.0,1,1,1
4,FL.BBN.01,2023-01-01 01:00:00,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,1.0,0,0,0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,72.0,168.0,0.0,BBN,...,0.0,0.00372,0.28,NaN,0.026415,0.056604,1.918889,0.060370,0.0,0.068966,0.0,0.0,0.0,4.0,2.936,11.0,0.316,0.0,0.258819,0.965926,0.017202,0.999852,0,0.739791,0.672836,0.153233,0.0,1,1,1


In [3]:
feats = feature_columns(sample)
labels = label_columns(sample)
print(f"{len(feats)} features:")
for group in ["fl_", "rain_", "water_", "flow_", "cal_", "tide_", "elev", "slope",
              "depression", "dist_"]:
    got = [f for f in feats if f.startswith(group)]
    if got:
        print(f"  {group:12s} {len(got):2d}  {', '.join(got)}")
print(f"\n{len(labels)} labels: {', '.join(labels[:6])} ...")

50 features:
  fl_          12  fl_depth_now, fl_depth_lag1h, fl_depth_lag3h, fl_rise_15min, fl_rise_1h, fl_max3h, fl_max24h, fl_mean1h, fl_std3h, fl_hrs_since_5cm, fl_hrs_since_15cm, fl_missing_share3h
  rain_        15  rain_rf1hr_mean, rain_rf1hr_max, rain_rf3hr_mean, rain_rf3hr_max, rain_rf6hr_mean, rain_rf24hr_mean, rain_gauges_reporting, rain_rf1hr_delta1h, rain_rf1hr_delta3h, rain_spread, rain_antecedent_ratio, rain_fcst_1h, rain_fcst_3h, rain_fcst_6h, rain_x_recent_flood
  water_        5  water_rise1h_mean, water_rise1h_max, water_rise3h_mean, water_rising_share, water_offline_share
  flow_         4  flow_mean, flow_velocity_mean, flow_negative_share, flow_offline_share
  cal_          5  cal_hour_sin, cal_hour_cos, cal_doy_sin, cal_doy_cos, cal_monsoon
  tide_         3  tide_m2_sin, tide_m2_cos, tide_spring_neap
  elev          2  elev_m, elev_rank_local
  slope         1  slope_deg
  depression    1  depression_m
  dist_         1  dist_water_km

15 labels: y_maxdepth_1h, 

## 2. Leakage checks

Three checks. Each one has caught a real bug at some point, and each takes seconds.

In [4]:
# --- Check 1: does any feature correlate suspiciously with its own future label?
# A correlation near 1.0 means the answer leaked into the question. fl_depth_now will
# be high (0.5-0.7) and that is legitimate — water now genuinely predicts water soon.
# Anything above ~0.95 is a bug.
target = "y_maxdepth_1h"
numeric = sample[feats].select_dtypes("number")
corr = numeric.corrwith(sample[target]).abs().sort_values(ascending=False)
print(corr.head(12).round(3).to_string())
suspect = corr[corr > 0.95]
print(f"\nSuspiciously high (>0.95): {list(suspect.index) if len(suspect) else 'none'}")

fl_depth_now           0.744
fl_mean1h              0.614
rain_x_recent_flood    0.536
fl_max3h               0.468
fl_std3h               0.424
fl_depth_lag1h         0.387
fl_rise_1h             0.373
rain_rf1hr_mean        0.292
rain_rf1hr_max         0.292
rain_rf3hr_max         0.247
rain_rf3hr_mean        0.244
fl_rise_15min          0.224

Suspiciously high (>0.95): none


In [5]:
# --- Check 2: labels must describe the FUTURE, features the PRESENT.
# Take a station, find a moment where the label is 1, and read the raw depth series
# around it by hand. This is slower than a correlation but it is the check that
# actually convinces you.
tier, horizon = CFG["flood_event"]["primary_tier_cm"], 1
hit = sample[sample[f"y_ge{tier}_{horizon}h"] == 1]
if len(hit):
    row = hit.iloc[len(hit) // 2]
    code_, ts = row["station_code"], row["site_timestamp"]
    print(f"Station {code_} at {ts}: depth now = {row['fl_depth_now']}, "
          f"label says flooding within {horizon}h")

    import pyarrow.parquet as pq
    raw = pq.read_table(PATHS.parquet / "flood" / "2023.parquet",
                        columns=["station_code", "site_timestamp", "flood"]).to_pandas()
    raw["station_code"] = raw["station_code"].astype(str)
    window = raw[(raw["station_code"] == str(code_))
                 & (raw["site_timestamp"] >= ts - pd.Timedelta(minutes=30))
                 & (raw["site_timestamp"] <= ts + pd.Timedelta(hours=horizon))]
    print(window.to_string(index=False))
    print("\nExpected: depth at/below the tier at the anchor time, and reaching "
          f"{tier} cm somewhere in the following {horizon}h.")
else:
    print("No positive rows in this sample — try a wetter year.")

Station FL.LSI.04 at 2023-10-28 12:30:00: depth now = 0.0, label says flooding within 1h


station_code      site_timestamp  flood
   FL.LSI.04 2023-10-28 12:00:00    0.0
   FL.LSI.04 2023-10-28 12:05:00    0.0
   FL.LSI.04 2023-10-28 12:10:00    0.0
   FL.LSI.04 2023-10-28 12:15:00    0.0
   FL.LSI.04 2023-10-28 12:20:00    0.0
   FL.LSI.04 2023-10-28 12:25:00    0.0
   FL.LSI.04 2023-10-28 12:30:00    0.0
   FL.LSI.04 2023-10-28 12:35:00    0.0
   FL.LSI.04 2023-10-28 12:40:00   11.7
   FL.LSI.04 2023-10-28 12:45:00   14.8
   FL.LSI.04 2023-10-28 12:50:00   15.8
   FL.LSI.04 2023-10-28 12:55:00   17.3
   FL.LSI.04 2023-10-28 13:00:00   16.7
   FL.LSI.04 2023-10-28 13:05:00   16.5
   FL.LSI.04 2023-10-28 13:10:00   15.2
   FL.LSI.04 2023-10-28 13:15:00   16.9
   FL.LSI.04 2023-10-28 13:20:00   14.8
   FL.LSI.04 2023-10-28 13:25:00   11.8
   FL.LSI.04 2023-10-28 13:30:00   11.7

Expected: depth at/below the tier at the anchor time, and reaching 15 cm somewhere in the following 1h.


In [6]:
# --- Check 3: timestamps must be strictly increasing within each station.
# Out-of-order rows would make every trailing window silently wrong.
ordering = sample.sort_values(["station_code", "site_timestamp"])
bad = ordering.groupby("station_code", observed=True)["site_timestamp"].apply(
    lambda s: (s.diff().dropna() <= pd.Timedelta(0)).sum()).sum()
print(f"Out-of-order timestamps: {bad}   (must be 0)")

gaps = ordering.groupby("station_code", observed=True)["site_timestamp"].diff().dropna()
print(f"Anchor spacing — expected {CFG['forecast']['anchor_cadence_min']} min")
print(gaps.value_counts().head(3).to_string())

Out-of-order timestamps: 0   (must be 0)


Anchor spacing — expected 15 min
site_timestamp
0 days 00:15:00    3370249
0 days 00:30:00          9
0 days 01:15:00          3


## 3. How rare are the labels, and how many are real forecasts?

`describe_labels` splits every positive into onset (station currently dry — a genuine
forecast) and ongoing (already flooded — persistence gets it for free).

In [7]:
rows = []
for tier in CFG["flood_event"]["tiers_cm"].values():
    for horizon in CFG["forecast"]["horizons_h"]:
        rows.append(describe_labels(sample, tier, horizon))
pd.DataFrame(rows)

,label,rows,positives,positive_rate,onset_positives,ongoing_positives,onset_share_of_positives
0,y_ge5_1h,3370379,2879,0.000854,1555,1324,0.5401
1,y_ge5_3h,3370379,5910,0.001754,4581,1329,0.7751
2,y_ge5_6h,3370379,10390,0.003083,9061,1329,0.8721
3,y_ge15_1h,3370379,823,0.000244,398,425,0.4836
4,y_ge15_3h,3370379,1615,0.000479,1190,425,0.7368
5,y_ge15_6h,3370379,2803,0.000832,2378,425,0.8484
6,y_ge30_1h,3370379,147,0.000044,73,74,0.4966
7,y_ge30_3h,3370379,283,0.000084,209,74,0.7385
8,y_ge30_6h,3370379,487,0.000144,413,74,0.8480


**What to look for.** `onset_share_of_positives` should be well under 1. That is not
a problem with the data — floods last longer than 15 minutes, so most positive rows are
mid-flood. It is a warning about *interpretation*: a recall number computed over all
positives is mostly measuring persistence.

## 4. Missing values, and why we keep them

LightGBM handles NaN natively and learns which side of a split missingness belongs on.
That is genuinely useful here, because a missing reading is informative — sensors drop out
during the worst weather, so an outage is weak evidence of trouble.

We therefore **do not impute** for the tree models. The sequence models in notebook 06 do
need imputation, and they get a missing-value indicator column alongside it so the
information is not thrown away.

In [8]:
missing = sample[feats].isna().mean().sort_values(ascending=False)
print("Most-missing features:")
print((100 * missing.head(12)).round(1).to_string())
print(f"\nFeatures with no missing values: {(missing == 0).sum()}/{len(missing)}")

Most-missing features:
fl_depth_lag3h        0.0
rain_rf1hr_delta3h    0.0
water_rise3h_mean     0.0
fl_rise_1h            0.0
fl_depth_lag1h        0.0
fl_rise_15min         0.0
rain_rf1hr_delta1h    0.0
water_rise1h_mean     0.0
water_rise1h_max      0.0
fl_depth_now          0.0
fl_mean1h             0.0
fl_max3h              0.0

Features with no missing values: 36/50


## 5. Build every year

This is the long run. Each year is written separately so an interruption costs one year,
not the whole pass.

In [9]:
import pyarrow as pa
import pyarrow.parquet as pq

REBUILD = True         # set True to actually run the full build

if REBUILD:
    for year in CFG["raw"]["years"]:
        out_path = PATHS.training / f"year_{year}.parquet"
        if out_path.exists():
            print(f"[{year}] already built, skipping")
            continue
        df = build_year(year)
        pq.write_table(pa.Table.from_pandas(df, preserve_index=False),
                       out_path, compression="zstd")
        print(f"[{year}] wrote {len(df):,} rows -> {out_path.name}")
else:
    print("REBUILD is False — set it to True to run the full build.")
    print("Existing yearly files:")
    for p in sorted(PATHS.training.glob("year_*.parquet")):
        print(f"  {p.name}  {p.stat().st_size / 1e6:.1f} MB")

[2019] loading flood depths ...


[2019] building rain / water / flow aggregates ...


[2019] 3,468,960 rows, y_ge15_1h positives: 1,184


[2019] wrote 3,468,960 rows -> year_2019.parquet
[2020] loading flood depths ...


[2020] building rain / water / flow aggregates ...


...

[2024] 3,451,352 rows, y_ge15_1h positives: 294


[2024] wrote 3,451,352 rows -> year_2024.parquet
[2025] loading flood depths ...


[2025] building rain / water / flow aggregates ...


[2025] 3,348,382 rows, y_ge15_1h positives: 881


[2025] wrote 3,348,382 rows -> year_2025.parquet


## 6. Record the feature contract

`features.json` is the agreement between training and serving. If the model expects 31
columns and the API sends 28, predictions are silently wrong rather than loudly broken —
so the list is written down, versioned, and checked at load time.

In [10]:
import json

meta = {
    "version": CFG["project"]["version"],
    "features": feats,
    "labels": labels,
    "cadence_min": CFG["forecast"]["anchor_cadence_min"],
    "horizons_h": CFG["forecast"]["horizons_h"],
    "tiers_cm": sorted(CFG["flood_event"]["tiers_cm"].values()),
    "flood_event_definition": CFG["flood_event"],
    "notes": [
        "Rain is joined per district code prefix (covers 100% of flood districts).",
        "Rain is a DISTRICT AVERAGE; Bangkok floods from localised cells. "
        "Radar rainfall is the highest-value missing input.",
        "Water and flow are citywide aggregates — their codes are canal names, "
        "not districts, so they cannot be joined locally without coordinates.",
        "Tide terms are reconstructed from lunar periods, not measured. They give "
        "phase, not height.",
        "NaN is expected and meaningful; LightGBM handles it natively. Sequence "
        "models impute and add an indicator column.",
        "is_onset_* columns are evaluation metadata, never model inputs.",
    ],
}
(PATHS.training / "features.json").write_text(json.dumps(meta, indent=2))
print(f"Wrote features.json — {len(feats)} features, {len(labels)} labels")

Wrote features.json — 50 features, 15 labels


**Done when:** `data/training/year_YYYY.parquet` exists for every year and
`features.json` matches them.

Optional next steps that measurably help: `03b_forecast_rain.ipynb` and
`03c_terrain_features.ipynb`. Then `04_train_baseline.ipynb`.